# CLIP4CAD-GFA v4.8.1 Training

**Anchored Staged Learning with Hierarchical Codebook**

## Key Insight
The v4.8 training was stuck because BRep encoder starts random with no meaningful anchor.
v4.8.1 solves this with **staged training**:

| Stage | Goal | Anchor |
|-------|------|--------|
| 0 | Anchor BRep to PC | PC (ShapeLLM pre-trained) |
| 1 | Add text + codebook | BRep now meaningful |
| 2 | Close gap + hard neg | All modalities aligned |

## Architecture Changes from v4.8
- **Hierarchical Codebook**: 672 codes (16 category + 128 type + 512 variant + 16 spatial)
- **Sparse Selection**: Threshold-based, ~10-50 active codes per sample
- **Position-Aware**: BRep positions used for grounding
- **Reconstruction**: Auxiliary BRep reconstruction loss

In [ ]:
# Cell 1: Imports and Setup
import sys
sys.path.insert(0, '..')

import os
import gc
import math
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm
import numpy as np
from pathlib import Path

# Clean memory
gc.collect()
torch.cuda.empty_cache()

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Configuration
# ═══════════════════════════════════════════════════════════════════════════════
# DATA PATHS
# ═══════════════════════════════════════════════════════════════════════════════
DATA_ROOT = Path("d:/Defect_Det/MMCAD/data")
EMBEDDINGS_DIR = DATA_ROOT / "embeddings"
ALIGNED_DIR = DATA_ROOT / "aligned"

# PC and Text files
PC_FILE = Path("c:/Users/User/Desktop/pc_embeddings_full.h5")
TEXT_FILE = Path("c:/Users/User/Desktop/text_embeddings.h5")

# Output
OUTPUT_DIR = Path("../outputs/gfa_v4_8_1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING HYPERPARAMETERS
# ═══════════════════════════════════════════════════════════════════════════════
BATCH_SIZE = 256

# Stage epochs
STAGE0_EPOCHS = 10    # Anchor BRep to PC
STAGE1_EPOCHS = 15    # Add text + codebook
STAGE2_EPOCHS = 10    # Close gap + hard negatives

# Learning rates per stage
STAGE0_LR = 5e-4      # Higher LR for initial learning
STAGE1_LR = 1e-4      # Standard LR
STAGE2_LR = 1e-5      # Lower LR for fine-tuning

MAX_GRAD_NORM = 1.0

print("Configuration:")
print(f"  Data root: {DATA_ROOT}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Stage 0: {STAGE0_EPOCHS} epochs (anchor to PC)")
print(f"  Stage 1: {STAGE1_EPOCHS} epochs (text + codebook)")
print(f"  Stage 2: {STAGE2_EPOCHS} epochs (gap closing)")
print(f"  Output: {OUTPUT_DIR}")

In [ ]:
# Cell 3: Import Dataset
from clip4cad.data.gfa_dataset import GFAMappedDataset, gfa_collate_fn

print("Using GFAMappedDataset with use_autobrep=True")

In [ ]:
# Cell 4: Load Datasets
gc.collect()
torch.cuda.empty_cache()

print("Loading datasets with AutoBrep topology...")
print("=" * 60)

MAX_TRAIN_SAMPLES = 111000

# Train dataset - LOAD TO RAM
print(f"\n[1/2] Loading TRAIN dataset (max {MAX_TRAIN_SAMPLES:,} samples)...")
train_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="train",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    mapping_dir=str(ALIGNED_DIR),
    load_to_memory=True,
    use_live_text=False,
    use_autobrep=True,
    autobrep_dir=str(EMBEDDINGS_DIR),
    max_samples=MAX_TRAIN_SAMPLES,
)
print(f"Train: {len(train_dataset):,} samples")

# Val dataset - ON DISK
print("\n[2/2] Loading VAL dataset...")
val_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="val",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    mapping_dir=str(ALIGNED_DIR),
    load_to_memory=False,
    use_live_text=False,
    use_autobrep=True,
    autobrep_dir=str(EMBEDDINGS_DIR),
)
print(f"Val: {len(val_dataset):,} samples")

print("\n" + "=" * 60)
print("DATASETS READY!")

In [ ]:
# Cell 5: Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=gfa_collate_fn,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=gfa_collate_fn,
    pin_memory=True,
)

print(f"Train loader: {len(train_loader)} batches")
print(f"Val loader: {len(val_loader)} batches")

In [ ]:
# Cell 6: Batch Remapping Function
def remap_batch(batch):
    """Remap collate output keys to model expected keys."""
    remapped = {}
    for k, v in batch.items():
        # Remove 'brep_' prefix for model compatibility
        if k.startswith('brep_'):
            new_key = k[5:]  # Remove 'brep_' prefix
            remapped[new_key] = v
        else:
            remapped[k] = v
    
    # Split pc_features into local and global
    if 'pc_features' in remapped:
        pc = remapped.pop('pc_features')
        remapped['pc_local_features'] = pc[:, :-1, :]   # (B, N, 1024)
        remapped['pc_global_features'] = pc[:, -1, :]   # (B, 1024)
    
    # Rename text keys
    if 'desc_embedding' in remapped:
        remapped['text_features'] = remapped.pop('desc_embedding')
    if 'desc_mask' in remapped:
        remapped['text_mask'] = remapped.pop('desc_mask')
    
    return remapped

# Test remapping
test_batch = next(iter(train_loader))
test_batch = remap_batch(test_batch)
print("Remapped batch keys:")
for k, v in test_batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

In [ ]:
# Cell 7: Create Model
gc.collect()
torch.cuda.empty_cache()

from clip4cad.models.clip4cad_gfa_v4_8_1 import CLIP4CAD_GFA_v481, GFAv481Config
from clip4cad.losses.gfa_v4_8_1_losses import (
    GFAv481Loss,
    compute_modality_gap,
    compute_true_pair_cosine,
    compute_brep_pc_metrics,
    compute_code_diversity,
    compute_active_codes,
    mine_hard_negatives_by_code,
)

# Model configuration
config = GFAv481Config(
    d_face=48,
    d_edge=12,
    d_pc=1024,
    d_text=3072,
    d=256,
    d_proj=128,
    # Hierarchical codebook
    n_category=16,
    n_type_per_cat=8,
    n_variant_per_type=4,
    n_spatial=16,
    code_sparsity=0.1,
    # Architecture
    num_msg_layers=3,
    num_brep_tf_layers=4,
    num_heads=8,
    dropout=0.1,
)

model = CLIP4CAD_GFA_v481(config).to(device)
print(f"Model parameters: {model.count_parameters():,}")
print(f"Trainable parameters: {model.count_parameters(trainable_only=True):,}")
print(f"Total codes: {model.codebook.total_codes}")

In [ ]:
# Cell 8: Loss Function
criterion = GFAv481Loss(
    lambda_recon=0.5,
    lambda_align=0.5,
    lambda_uniform=0.3,
    lambda_code=0.3,
    lambda_diversity=0.1,
    lambda_hard_neg=0.0,  # Will be enabled in Stage 2
)

# Scaler for FP16
scaler = GradScaler()

print("Loss function initialized")

In [ ]:
# Cell 9: Verify Stage 0 Forward Pass
print("Verifying Stage 0 forward pass (BRep-PC only)...")

model.eval()
with torch.no_grad():
    test_batch = remap_batch(next(iter(train_loader)))
    with autocast():
        outputs = model.forward_stage0(test_batch)

print("\nStage 0 Outputs:")
for k, v in outputs.items():
    if isinstance(v, torch.Tensor):
        nan_count = v.isnan().sum().item()
        status = "NaN!" if nan_count > 0 else "OK"
        print(f"  {k}: {v.shape} [{status}]")
    else:
        print(f"  {k}: {v}")

# Test loss
loss, loss_dict = criterion(outputs, stage=0)
print("\nLoss components:")
for k, v in loss_dict.items():
    print(f"  {k}: {v.item():.4f}")

## Stage 0: Anchor BRep to PC

**Goal:** Make BRep encoder produce meaningful features by aligning to pre-trained PC encoder.

**Why this works:**
- ShapeLLM (PC encoder) is pre-trained and produces meaningful features
- PC and BRep represent the SAME 3D geometry
- BRep encoder has a stable target to learn toward

**Expected:** BRep-PC gap → 0, cosine → 0.7+

In [ ]:
# Cell 10: Stage 0 Training
print("\n" + "="*70)
print("STAGE 0: Anchoring BRep encoder to PC (ShapeLLM anchor)")
print("="*70)

optimizer = AdamW(model.parameters(), lr=STAGE0_LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=STAGE0_EPOCHS)

stage0_history = {
    'loss': [], 'gap': [], 'cosine': [], 'recon': []
}

for epoch in range(1, STAGE0_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    epoch_metrics = {'gap': [], 'cos': [], 'recon': []}
    
    pbar = tqdm(train_loader, desc=f"Stage 0, Epoch {epoch}")
    
    for batch in pbar:
        batch = remap_batch(batch)
        
        with autocast():
            outputs = model.forward_stage0(batch)
            loss, loss_dict = criterion(outputs, stage=0)
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss_dict['total'].item()
        
        with torch.no_grad():
            gap, cos = compute_brep_pc_metrics(outputs['z_brep_raw'], outputs['z_pc_raw'])
            epoch_metrics['gap'].append(gap)
            epoch_metrics['cos'].append(cos)
            epoch_metrics['recon'].append(loss_dict['recon'].item())
        
        pbar.set_postfix({
            'loss': f"{loss_dict['total'].item():.3f}",
            'gap': f"{gap:.3f}",
            'cos': f"{cos:.3f}",
        })
    
    scheduler.step()
    
    # Epoch summary
    avg_loss = epoch_loss / len(train_loader)
    avg_gap = np.mean(epoch_metrics['gap'])
    avg_cos = np.mean(epoch_metrics['cos'])
    avg_recon = np.mean(epoch_metrics['recon'])
    
    stage0_history['loss'].append(avg_loss)
    stage0_history['gap'].append(avg_gap)
    stage0_history['cosine'].append(avg_cos)
    stage0_history['recon'].append(avg_recon)
    
    print(f"Epoch {epoch}: Loss={avg_loss:.4f}, Gap={avg_gap:.4f}, Cos={avg_cos:.4f}, Recon={avg_recon:.4f}")

# Save Stage 0 checkpoint
torch.save({
    'epoch': STAGE0_EPOCHS,
    'model_state_dict': model.state_dict(),
    'history': stage0_history,
}, OUTPUT_DIR / 'checkpoint_stage0.pt')

print("\nStage 0 complete! BRep encoder anchored to PC.")
print(f"Final BRep-PC gap: {stage0_history['gap'][-1]:.4f}")
print(f"Final BRep-PC cosine: {stage0_history['cosine'][-1]:.4f}")

## Stage 1: Add Text + Codebook

**Goal:** Learn codebook structure and establish relative alignment.

**Process:**
1. Initialize codebook from text encoder (K-means)
2. Enable full forward pass with HierarchicalCodebookGrounding
3. Train with 3-way contrastive + code alignment

**No gap closing yet** - let contrastive establish relative positions first.

In [ ]:
# Cell 11: Initialize Codebook
print("\n" + "="*70)
print("Initializing hierarchical codebook from text encoder...")
print("="*70)

model.initialize_codebook(train_loader, device, remap_fn=remap_batch, max_batches=50)

print(f"\nCodebook initialized!")
print(f"  Total codes: {model.codebook.total_codes}")
print(f"  Temperature: {model.codebook.tau.item():.4f}")

In [ ]:
# Cell 12: Verify Stage 1 Forward Pass
print("Verifying Stage 1 forward pass (full with codebook)...")

model.eval()
with torch.no_grad():
    test_batch = remap_batch(next(iter(train_loader)))
    with autocast():
        outputs = model(test_batch, stage=1)

print("\nStage 1 Outputs:")
for k, v in outputs.items():
    if isinstance(v, torch.Tensor):
        nan_count = v.isnan().sum().item()
        status = "NaN!" if nan_count > 0 else "OK"
        print(f"  {k}: {v.shape} [{status}]")
    elif isinstance(v, dict):
        print(f"  {k}: {{...}}")
    else:
        print(f"  {k}: {v}")

# Check active codes
active_codes = compute_active_codes(outputs['w_text'], outputs['w_brep'], outputs['w_pc'])
print("\nActive codes per level (avg):")
for level in ['category', 'type', 'variant', 'spatial']:
    print(f"  {level}: {active_codes[f'{level}_avg']:.1f}")

# Test loss
loss, loss_dict = criterion(outputs, stage=1)
print("\nLoss components:")
for k, v in loss_dict.items():
    print(f"  {k}: {v.item():.4f}")

In [ ]:
# Cell 13: Stage 1 Training
print("\n" + "="*70)
print("STAGE 1: Adding text and codebook alignment")
print("="*70)

# Reset optimizer with lower LR
optimizer = AdamW(model.parameters(), lr=STAGE1_LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=STAGE1_EPOCHS)

stage1_history = {
    'loss': [], 'gap_brep': [], 'gap_pc': [],
    'cos_brep': [], 'cos_pc': [], 'diversity': [],
    'contrastive': [], 'code': [],
}

for epoch in range(1, STAGE1_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    epoch_metrics = {
        'gap_brep': [], 'gap_pc': [],
        'cos_brep': [], 'cos_pc': [],
        'diversity': [], 'contrastive': [], 'code': []
    }
    
    pbar = tqdm(train_loader, desc=f"Stage 1, Epoch {epoch}")
    
    for batch in pbar:
        batch = remap_batch(batch)
        
        with autocast():
            outputs = model(batch, stage=1)
            loss, loss_dict = criterion(outputs, stage=1)
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss_dict['total'].item()
        
        with torch.no_grad():
            gap_brep, gap_pc = compute_modality_gap(
                outputs['z_text_raw'], outputs['z_brep_raw'], outputs['z_pc_raw']
            )
            cos_brep, cos_pc = compute_true_pair_cosine(
                outputs['z_text_raw'], outputs['z_brep_raw'], outputs['z_pc_raw']
            )
            diversity = compute_code_diversity(
                outputs['w_text'], outputs['w_brep'], outputs['w_pc'], level='category'
            )
            
            epoch_metrics['gap_brep'].append(gap_brep)
            epoch_metrics['gap_pc'].append(gap_pc)
            epoch_metrics['cos_brep'].append(cos_brep)
            epoch_metrics['cos_pc'].append(cos_pc)
            epoch_metrics['diversity'].append(diversity)
            epoch_metrics['contrastive'].append(loss_dict['contrastive'].item())
            epoch_metrics['code'].append(loss_dict['code'].item())
        
        pbar.set_postfix({
            'loss': f"{loss_dict['total'].item():.3f}",
            'gap': f"{gap_brep:.2f}",
            'cos': f"{cos_brep:.3f}",
            'div': f"{diversity:.3f}",
        })
    
    scheduler.step()
    
    # Epoch summary
    avg_loss = epoch_loss / len(train_loader)
    for k in epoch_metrics:
        stage1_history[k].append(np.mean(epoch_metrics[k]))
    stage1_history['loss'].append(avg_loss)
    
    print(f"Epoch {epoch}: Loss={avg_loss:.4f}, Gap(B)={np.mean(epoch_metrics['gap_brep']):.4f}, "
          f"Cos(B)={np.mean(epoch_metrics['cos_brep']):.4f}, Div={np.mean(epoch_metrics['diversity']):.4f}")

# Save Stage 1 checkpoint
torch.save({
    'epoch': STAGE0_EPOCHS + STAGE1_EPOCHS,
    'model_state_dict': model.state_dict(),
    'stage0_history': stage0_history,
    'stage1_history': stage1_history,
}, OUTPUT_DIR / 'checkpoint_stage1.pt')

print("\nStage 1 complete! Text and codebook aligned.")

## Stage 2: Gap Closing + Hard Negatives

**Goal:** Close the absolute modality gap and improve fine-grained discrimination.

**New losses:**
- L_ATP: Align True Pairs (closes the gap!)
- L_CU: Centroid Uniformity (prevents collapse)
- L_hard_neg: Hard negative mining

**Lower LR:** Fine-tuning, not breaking learned representations.

In [ ]:
# Cell 14: Mine Hard Negatives
print("\n" + "="*70)
print("Mining hard negatives before Stage 2...")
print("="*70)

hard_negatives = mine_hard_negatives_by_code(model, train_loader, device, top_k=10, max_batches=50)
print(f"Mined {len(hard_negatives)} hard negative sets")

# Enable hard negative loss
criterion.lambda_hard_neg = 0.3

In [ ]:
# Cell 15: Stage 2 Training
print("\n" + "="*70)
print("STAGE 2: Gap closing and hard negative mining")
print("="*70)

# Reset optimizer with even lower LR
optimizer = AdamW(model.parameters(), lr=STAGE2_LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=STAGE2_EPOCHS)

stage2_history = {
    'loss': [], 'gap_brep': [], 'gap_pc': [],
    'cos_brep': [], 'cos_pc': [], 'diversity': [],
    'contrastive': [], 'align': [], 'uniform': [], 'code': [],
}

best_gap = float('inf')

for epoch in range(1, STAGE2_EPOCHS + 1):
    # Re-mine hard negatives every 3 epochs
    if epoch > 1 and epoch % 3 == 0:
        print("Re-mining hard negatives...")
        hard_negatives = mine_hard_negatives_by_code(model, train_loader, device, top_k=10, max_batches=50)
    
    model.train()
    epoch_loss = 0.0
    epoch_metrics = {
        'gap_brep': [], 'gap_pc': [],
        'cos_brep': [], 'cos_pc': [],
        'diversity': [], 'contrastive': [], 'align': [], 'uniform': [], 'code': []
    }
    
    pbar = tqdm(train_loader, desc=f"Stage 2, Epoch {epoch}")
    
    for batch_idx, batch in enumerate(pbar):
        batch = remap_batch(batch)
        
        # Get hard negatives for this batch
        batch_size = batch['face_features'].shape[0]
        start_idx = batch_idx * batch_size
        batch_hard_negs = [
            hard_negatives.get(start_idx + i) for i in range(batch_size)
        ]
        
        with autocast():
            outputs = model(batch, stage=2)
            loss, loss_dict = criterion(outputs, stage=2, hard_negatives=batch_hard_negs)
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss_dict['total'].item()
        
        with torch.no_grad():
            gap_brep, gap_pc = compute_modality_gap(
                outputs['z_text_raw'], outputs['z_brep_raw'], outputs['z_pc_raw']
            )
            cos_brep, cos_pc = compute_true_pair_cosine(
                outputs['z_text_raw'], outputs['z_brep_raw'], outputs['z_pc_raw']
            )
            diversity = compute_code_diversity(
                outputs['w_text'], outputs['w_brep'], outputs['w_pc'], level='category'
            )
            
            epoch_metrics['gap_brep'].append(gap_brep)
            epoch_metrics['gap_pc'].append(gap_pc)
            epoch_metrics['cos_brep'].append(cos_brep)
            epoch_metrics['cos_pc'].append(cos_pc)
            epoch_metrics['diversity'].append(diversity)
            epoch_metrics['contrastive'].append(loss_dict['contrastive'].item())
            epoch_metrics['align'].append(loss_dict['align'].item())
            epoch_metrics['uniform'].append(loss_dict['uniform'].item())
            epoch_metrics['code'].append(loss_dict['code'].item())
        
        pbar.set_postfix({
            'loss': f"{loss_dict['total'].item():.3f}",
            'gap': f"{gap_brep:.2f}",
            'cos': f"{cos_brep:.3f}",
        })
    
    scheduler.step()
    
    # Epoch summary
    avg_loss = epoch_loss / len(train_loader)
    avg_gap = np.mean(epoch_metrics['gap_brep'])
    
    for k in epoch_metrics:
        stage2_history[k].append(np.mean(epoch_metrics[k]))
    stage2_history['loss'].append(avg_loss)
    
    if avg_gap < best_gap:
        best_gap = avg_gap
        torch.save({
            'epoch': STAGE0_EPOCHS + STAGE1_EPOCHS + epoch,
            'model_state_dict': model.state_dict(),
            'best_gap': best_gap,
        }, OUTPUT_DIR / 'checkpoint_best.pt')
        print(f"  New best gap: {best_gap:.4f}!")
    
    print(f"Epoch {epoch}: Loss={avg_loss:.4f}, Gap(B)={np.mean(epoch_metrics['gap_brep']):.4f}, "
          f"Cos(B)={np.mean(epoch_metrics['cos_brep']):.4f}, Align={np.mean(epoch_metrics['align']):.4f}")

print("\n" + "="*70)
print("Training Complete!")
print(f"Best gap: {best_gap:.4f}")

In [ ]:
# Cell 16: Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Stage 0: BRep-PC metrics
ax = axes[0, 0]
ax.plot(stage0_history['gap'], label='BRep-PC Gap')
ax.plot(stage0_history['cosine'], label='BRep-PC Cosine')
ax.set_xlabel('Epoch')
ax.set_title('Stage 0: BRep-PC Anchoring')
ax.legend()
ax.grid(True, alpha=0.3)

# Stage 1+2: Modality Gap
ax = axes[0, 1]
s1_epochs = list(range(1, len(stage1_history['gap_brep']) + 1))
s2_epochs = list(range(len(s1_epochs) + 1, len(s1_epochs) + len(stage2_history['gap_brep']) + 1))
ax.plot(s1_epochs, stage1_history['gap_brep'], label='Stage 1 - BRep')
ax.plot(s2_epochs, stage2_history['gap_brep'], label='Stage 2 - BRep')
ax.axhline(y=0, color='g', linestyle='--', alpha=0.5, label='Target (0)')
ax.axvline(x=len(s1_epochs), color='r', linestyle='--', alpha=0.3)
ax.set_xlabel('Epoch')
ax.set_ylabel('Gap')
ax.set_title('Modality Gap (should \u2192 0)')
ax.legend()
ax.grid(True, alpha=0.3)

# True-Pair Cosine
ax = axes[0, 2]
ax.plot(s1_epochs, stage1_history['cos_brep'], label='Stage 1')
ax.plot(s2_epochs, stage2_history['cos_brep'], label='Stage 2')
ax.axhline(y=1.0, color='g', linestyle='--', alpha=0.5, label='Target (1.0)')
ax.axvline(x=len(s1_epochs), color='r', linestyle='--', alpha=0.3)
ax.set_xlabel('Epoch')
ax.set_ylabel('Cosine Similarity')
ax.set_title('True-Pair Cosine (should \u2192 1.0)')
ax.legend()
ax.grid(True, alpha=0.3)

# Loss
ax = axes[1, 0]
all_loss = stage0_history['loss'] + stage1_history['loss'] + stage2_history['loss']
ax.plot(all_loss)
ax.axvline(x=STAGE0_EPOCHS, color='r', linestyle='--', alpha=0.3, label='Stage 1')
ax.axvline(x=STAGE0_EPOCHS + STAGE1_EPOCHS, color='r', linestyle='--', alpha=0.3, label='Stage 2')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.grid(True, alpha=0.3)

# Diversity
ax = axes[1, 1]
all_div = stage1_history['diversity'] + stage2_history['diversity']
ax.plot(all_div)
ax.axhline(y=0.8, color='g', linestyle='--', alpha=0.5, label='Target (>0.8)')
ax.set_xlabel('Epoch (Stage 1+2)')
ax.set_ylabel('Diversity')
ax.set_title('Code Diversity')
ax.legend()
ax.grid(True, alpha=0.3)

# Summary
axes[1, 2].axis('off')
summary_text = (
    f"GFA v4.8.1 Training Summary\n\n"
    f"Stage 0 (Anchor): {STAGE0_EPOCHS} epochs\n"
    f"  BRep-PC Cosine: {stage0_history['cosine'][-1]:.4f}\n\n"
    f"Stage 1 (Align): {STAGE1_EPOCHS} epochs\n"
    f"  Gap: {stage1_history['gap_brep'][-1]:.4f}\n\n"
    f"Stage 2 (Close): {STAGE2_EPOCHS} epochs\n"
    f"  Best Gap: {best_gap:.4f}\n"
    f"  Final Cosine: {stage2_history['cos_brep'][-1]:.4f}\n"
    f"  Diversity: {stage2_history['diversity'][-1]:.4f}"
)
axes[1, 2].text(0.5, 0.5, summary_text, ha='center', va='center',
                fontsize=12, family='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()

In [ ]:
# Cell 17: Save Final Model
torch.save({
    'config': config,
    'model_state_dict': model.state_dict(),
    'best_gap': best_gap,
    'stage0_history': stage0_history,
    'stage1_history': stage1_history,
    'stage2_history': stage2_history,
}, OUTPUT_DIR / 'gfa_v4_8_1_final.pt')

print(f"Final model saved to {OUTPUT_DIR / 'gfa_v4_8_1_final.pt'}")
print(f"Best modality gap: {best_gap:.4f}")

In [ ]:
# Cell 18: Retrieval Evaluation
print("Testing retrieval performance...")

model.eval()
all_z_text = []
all_z_brep = []
all_z_pc = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Encoding"):
        batch = remap_batch(batch)
        with autocast():
            outputs = model(batch, stage=2)
        
        all_z_text.append(outputs['z_text'].cpu())
        all_z_brep.append(outputs['z_brep'].cpu())
        all_z_pc.append(outputs['z_pc'].cpu())

z_text = torch.cat(all_z_text, dim=0)
z_brep = torch.cat(all_z_brep, dim=0)
z_pc = torch.cat(all_z_pc, dim=0)

# Normalize
z_text = F.normalize(z_text, dim=-1)
z_brep = F.normalize(z_brep, dim=-1)
z_pc = F.normalize(z_pc, dim=-1)

N = z_text.shape[0]
print(f"\nEvaluation set: {N} samples")

# Text -> BRep retrieval
sim_t2b = z_text @ z_brep.T
ranks_t2b = (sim_t2b.argsort(dim=1, descending=True) == torch.arange(N).unsqueeze(1)).float().argmax(dim=1)
r1_t2b = (ranks_t2b < 1).float().mean().item() * 100
r5_t2b = (ranks_t2b < 5).float().mean().item() * 100
r10_t2b = (ranks_t2b < 10).float().mean().item() * 100

# Text -> PC retrieval
sim_t2p = z_text @ z_pc.T
ranks_t2p = (sim_t2p.argsort(dim=1, descending=True) == torch.arange(N).unsqueeze(1)).float().argmax(dim=1)
r1_t2p = (ranks_t2p < 1).float().mean().item() * 100
r5_t2p = (ranks_t2p < 5).float().mean().item() * 100

# BRep -> PC retrieval (geometry-only)
sim_b2p = z_brep @ z_pc.T
ranks_b2p = (sim_b2p.argsort(dim=1, descending=True) == torch.arange(N).unsqueeze(1)).float().argmax(dim=1)
r1_b2p = (ranks_b2p < 1).float().mean().item() * 100
r5_b2p = (ranks_b2p < 5).float().mean().item() * 100

print("\n" + "="*50)
print("Retrieval Results:")
print("="*50)
print(f"Text -> BRep: R@1={r1_t2b:.1f}%, R@5={r5_t2b:.1f}%, R@10={r10_t2b:.1f}%")
print(f"Text -> PC:   R@1={r1_t2p:.1f}%, R@5={r5_t2p:.1f}%")
print(f"BRep -> PC:   R@1={r1_b2p:.1f}%, R@5={r5_b2p:.1f}%")
print("="*50)